In [1]:
import pandas as pd
import plotly.graph_objects as go

# file_path = "dados-tanque-cônico.csv"
file_path = "https://docs.google.com/spreadsheets/d/1UPq-_KwH0DZXkwSryfIc5ePeHTOPsQVSuMVaHeimwSI/export?format=csv&gid=0"

arquivo = pd.read_csv(file_path, decimal=",")
arquivo.head()

t = arquivo["t"].to_numpy()
Q = arquivo["Qi(t)"].to_numpy()
h_exp = arquivo["H(t)"].to_numpy()


fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=Q, mode="lines", name="Q(t)"))
fig.add_trace(go.Scatter(x=t, y=h_exp, mode="lines", name="h(t)"))
fig.update_layout(title="Gráfico de q e h em função de t", xaxis_title="t")


In [2]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d

H = 25
R = 15
Cv = 8.5
Q_interp = interp1d(t, Q, kind="linear")


def edo(t, y):
    h = y[0]
    Q = Q_interp(t)

    if h <= 0:
        print(f"t = {t} s, h = {h} m")
        h = 0.1

    dhdt = ((H**2) * (Q - Cv * np.sqrt(h))) / (np.pi * (R**2) * (h**2))
    return [dhdt]


def edo_linear(t, y):
    h_barra = y[0]
    Q = Q_interp(t)

    h = h_barra + h_exp[0]
    Q_barra = Q - Q_interp(0)

    if h <= 0:
        print(f"t = {t} s, h = {h} m")
        h = 0.1

    ah = (-2 * (H**2) * Q) / (np.pi * (R**2) * (h**3)) + (3 * (H**2) * Cv) / (
        2 * np.pi * (R**3) * (h ** (5 / 2))
    )
    aq = (H**2) / (np.pi * (R**2) * (h**2))

    dh_barra_dt = ah * h_barra + aq * Q_barra
    return [dh_barra_dt]


t_np = np.arange(t[0], t[-1], 0.1)

sol = solve_ivp(edo, [t[0], t[-1]], [h_exp[0]], t_eval=t_np, method="LSODA")
h_model = sol.y[0]
sol_linear = solve_ivp(edo_linear, [t[0], t[-1]], [0], t_eval=t_np, method="LSODA")

h_linear = sol_linear.y[0] + h_exp[0]

fig2 = go.Figure()
# fig2.add_trace(go.Scatter(x=t, y=Q, mode="lines", name="Q(t)"))
# fig2.add_trace(go.Scatter(x=t_np, y=Q_interp(t_np), mode="lines", name="Q(t) interpolado"))
fig2.add_trace(go.Scatter(x=t, y=h_exp, mode="lines", name="h(t) experimental"))
fig2.add_trace(
    go.Scatter(x=sol.t, y=h_model, mode="lines", name="h(t) modelo original")
)
fig2.add_trace(
    go.Scatter(x=sol_linear.t, y=h_linear, mode="lines", name="h(t) modelo linear")
)
fig2.update_layout(xaxis_title="tempo / s", yaxis_title="nível / m")
